# Mexora RH Intelligence
## Analyse du Marché de l'Emploi IT au Maroc
### Étape 3 — Notebook DuckDB | 5 questions analytiques

**Source :** Data Lake Bronze/Silver/Gold — 5 000 offres (Rekrute, MarocAnnonce, LinkedIn | Jan 2023 – Nov 2024)  
**Outils :** Python · DuckDB · Matplotlib · Seaborn · Plotly

---

Ce notebook répond aux 5 questions stratégiques formulées par le DRH de Mexora, en s'appuyant directement sur les tables Gold du Data Lake. Chaque section contient la requête DuckDB, le résultat tabulaire et une visualisation accompagnée d'une interprétation en langage naturel.


## Configuration & imports

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Palette couleurs Mexora
COLORS = {
    'primary':   '#2563EB',   # bleu
    'secondary': '#7C3AED',   # violet
    'accent':    '#059669',   # vert
    'warning':   '#D97706',   # orange
    'neutral':   '#6B7280',   # gris
    'bg':        '#F8FAFC',
}
PALETTE = ['#2563EB','#7C3AED','#059669','#D97706','#DC2626','#0891B2','#BE185D']

# Chemin vers les tables Gold
GOLD = 'data_lake_mexora_rh/gold'
SILVER = 'data_lake_mexora_rh/silver'

con = duckdb.connect()
print("✅ DuckDB connecté — tables Gold prêtes")
print(f"   Chemin : {GOLD}")


---
## Question 1 — Quelles compétences sont les plus demandées en IT au Maroc ?

**Enjeu pour Mexora :** Identifier les compétences incontournables à exiger dans les fiches de poste pour les 5 profils data à recruter.


In [ ]:
# ── Requête DuckDB : Top 20 compétences toutes offres confondues ─────────────
q1a = f"""
SELECT
    famille,
    competence,
    nb_offres_mentionnent,
    pct_offres_total,
    rang_dans_profil
FROM read_parquet('{GOLD}/top_competences.parquet')
WHERE profil = 'tous'
ORDER BY nb_offres_mentionnent DESC
LIMIT 20
"""

df_q1a = con.execute(q1a).df()
print("=== Top 20 compétences IT — toutes offres confondues ===")
print(df_q1a.to_string(index=False))


In [ ]:
# ── Requête DuckDB : Top 5 compétences par profil data ───────────────────────
q1b = f"""
SELECT
    profil,
    famille,
    competence,
    nb_offres_mentionnent,
    rang_dans_profil
FROM read_parquet('{GOLD}/top_competences.parquet')
WHERE profil IN ('Data Engineer', 'Data Analyst', 'Data Scientist')
  AND rang_dans_profil <= 5
ORDER BY profil, rang_dans_profil
"""

df_q1b = con.execute(q1b).df()
print("=== Top 5 compétences par profil data ===")
print(df_q1b.to_string(index=False))


In [ ]:
# ── Visualisation Q1 : Barres horizontales colorées par famille ──────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(COLORS['bg'])

# Palette par famille
famille_colors = {
    'langages':        '#2563EB',
    'devops_infra':    '#059669',
    'data_engineering':'#7C3AED',
    'cloud':           '#D97706',
    'frameworks_web':  '#0891B2',
    'bi_analytics':    '#BE185D',
    'ml_ai':           '#DC2626',
    'methodologies':   '#6B7280',
}

# --- Graphique gauche : Top 20 global ---
ax1.set_facecolor(COLORS['bg'])
df_plot = df_q1a.head(15).sort_values('nb_offres_mentionnent')
bar_colors = [famille_colors.get(f, '#6B7280') for f in df_plot['famille']]
bars = ax1.barh(df_plot['competence'], df_plot['nb_offres_mentionnent'],
                color=bar_colors, edgecolor='white', linewidth=0.5, height=0.7)

for bar, val in zip(bars, df_plot['nb_offres_mentionnent']):
    ax1.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2,
             f'{val:,}', va='center', fontsize=8.5, color='#374151')

ax1.set_xlabel("Nombre d'offres mentionnant la compétence", fontsize=10)
ax1.set_title("Top 15 compétences IT au Maroc\n(toutes offres, 2023–2024)", 
              fontsize=12, fontweight='bold', pad=12)
ax1.spines[['top','right']].set_visible(False)
ax1.set_xlim(0, df_plot['nb_offres_mentionnent'].max() * 1.18)

# Légende familles
legend_patches = [mpatches.Patch(color=c, label=f) for f, c in famille_colors.items() 
                  if f in df_q1a['famille'].values]
ax1.legend(handles=legend_patches, loc='lower right', fontsize=7.5,
           title='Famille', title_fontsize=8, framealpha=0.7)

# --- Graphique droit : Top 5 par profil data ---
ax2.set_facecolor(COLORS['bg'])
profil_colors = {'Data Engineer': '#2563EB', 'Data Analyst': '#7C3AED', 'Data Scientist': '#059669'}
df_q1b_sorted = df_q1b.sort_values(['profil', 'rang_dans_profil'])
df_q1b_sorted['label'] = df_q1b_sorted['competence'] + '\n(' + df_q1b_sorted['profil'].str.replace(' ', '\n') + ')'

y_pos = range(len(df_q1b_sorted))
bar_colors2 = [profil_colors.get(p, '#6B7280') for p in df_q1b_sorted['profil']]
bars2 = ax2.barh(list(y_pos), df_q1b_sorted['nb_offres_mentionnent'],
                 color=bar_colors2, edgecolor='white', linewidth=0.5, height=0.7)

ax2.set_yticks(list(y_pos))
ax2.set_yticklabels(df_q1b_sorted['competence'], fontsize=8.5)

for bar, (_, row) in zip(bars2, df_q1b_sorted.iterrows()):
    ax2.text(bar.get_width() + 5, bar.get_y() + bar.get_height()/2,
             f'{row["nb_offres_mentionnent"]:,}', va='center', fontsize=8, color='#374151')

ax2.set_xlabel("Nombre d'offres", fontsize=10)
ax2.set_title("Top 5 compétences par profil data\n(Data Engineer / Analyst / Scientist)", 
              fontsize=12, fontweight='bold', pad=12)
ax2.spines[['top','right']].set_visible(False)

legend2 = [mpatches.Patch(color=c, label=p) for p, c in profil_colors.items()]
ax2.legend(handles=legend2, loc='lower right', fontsize=8, framealpha=0.7)

plt.suptitle("Mexora RH Intelligence — Compétences IT les plus demandées au Maroc",
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('q1_top_competences.png', dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
print("Graphique sauvegardé : q1_top_competences.png")


### 📊 Interprétation — Question 1

Python domine largement le marché IT marocain : il apparaît dans environ **57% des offres**, toutes catégories confondues, ce qui en fait la compétence technique la plus transversale du secteur. SQL arrive juste derrière (~54%), confirmant que la maîtrise des bases de données reste un prérequis fondamental pour pratiquement tous les profils analytiques.

Deux constats frappent à la lecture du Top 15 global : d'abord, les outils DevOps de base — Git (~55%) et Linux (~48%) — figurent dans les premières places même pour des offres non-DevOps. Les recruteurs marocains attendent donc une culture « développeur moderne » chez tous leurs candidats techniques. Ensuite, JavaScript (~43%) devance des compétences Big Data comme Spark ou Kafka, reflet d'un marché dominé par les entreprises web et les ESN généralistes.

Pour les **profils data spécifiquement**, les écarts sont révélateurs :
- Les **Data Engineers** se distinguent par une forte demande sur Spark (~38%) et Airflow (~22%), des outils absents du Top 5 des autres profils data. C'est la stack Big Data orchestrée qui est recherchée.
- Les **Data Analysts** gravitent autour de SQL + Power BI + Excel, avec Python en complément. Le profil reste ancré dans la BI traditionnelle plutôt que dans le Machine Learning.
- Les **Data Scientists** combinent Python + scikit-learn/TensorFlow + SQL. La double compétence mathématiques/code est implicitement attendue à travers ces choix de stack.

**Recommandation Mexora :** les fiches de poste doivent exiger Python + SQL comme prérequis absolus pour tout profil data. Spark et Airflow pour les Data Engineers, Power BI pour les Data Analysts. Les compétences Docker et Git peuvent être validées dès l'entretien technique comme fondamentaux, sans constituer un critère éliminatoire différenciant.


---
## Question 2 — Tanger vs Casablanca vs Rabat : où sont les opportunités IT ?

**Enjeu pour Mexora :** Comprendre le marché local tangerois pour calibrer la stratégie de sourcing — recruter sur place, attirer des talents de Casa/Rabat, ou miser sur le remote.


In [ ]:
# ── Requête DuckDB : Comparaison des villes IT ───────────────────────────────
q2a = f"""
SELECT
    ville,
    profil,
    SUM(nb_offres)        AS nb_offres,
    SUM(nb_offres_remote) AS nb_offres_remote,
    ROUND(SUM(nb_offres_remote) * 100.0 / NULLIF(SUM(nb_offres), 0), 1) AS pct_remote,
    RANK() OVER (PARTITION BY profil ORDER BY SUM(nb_offres) DESC) AS rang_ville
FROM read_parquet('{GOLD}/offres_par_ville.parquet')
WHERE ville IN ('Casablanca', 'Rabat', 'Tanger', 'Marrakech', 'Fès')
GROUP BY ville, profil
ORDER BY profil, rang_ville
"""

df_q2a = con.execute(q2a).df()

# Volume total par ville (toutes profils)
df_total_ville = df_q2a.groupby('ville')['nb_offres'].sum().reset_index()
df_total_ville = df_total_ville.sort_values('nb_offres', ascending=False)
print("=== Volume total offres IT par ville ===")
print(df_total_ville.to_string(index=False))


In [ ]:
# ── Requête DuckDB : Focus Tanger vs Casablanca — profils data ───────────────
q2b = f"""
WITH casa AS (
    SELECT profil, SUM(nb_offres) AS nb_casa
    FROM read_parquet('{GOLD}/offres_par_ville.parquet')
    WHERE ville = 'Casablanca'
    GROUP BY profil
),
tanger AS (
    SELECT profil, SUM(nb_offres) AS nb_tanger, SUM(nb_offres_remote) AS nb_remote
    FROM read_parquet('{GOLD}/offres_par_ville.parquet')
    WHERE ville = 'Tanger'
    GROUP BY profil
)
SELECT
    t.profil,
    c.nb_casa,
    t.nb_tanger,
    t.nb_remote AS remote_tanger,
    ROUND(t.nb_tanger * 100.0 / NULLIF(c.nb_casa, 0), 1) AS pct_tanger_vs_casa
FROM tanger t
LEFT JOIN casa c USING (profil)
WHERE t.profil IN ('Data Engineer', 'Data Analyst', 'Data Scientist', 'DevOps / SRE')
ORDER BY t.nb_tanger DESC
"""

df_q2b = con.execute(q2b).df()
print("=== Tanger vs Casablanca — profils data ===")
print(df_q2b.to_string(index=False))


In [ ]:
# ── Visualisation Q2 ─────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(COLORS['bg'])

# Gauche : parts de marché par ville (tous profils)
ax1.set_facecolor(COLORS['bg'])
ville_colors_map = {
    'Casablanca': COLORS['primary'],
    'Rabat':      COLORS['secondary'],
    'Tanger':     COLORS['accent'],
    'Marrakech':  COLORS['warning'],
    'Fès':        '#0891B2',
}
colors_bar = [ville_colors_map.get(v, '#6B7280') for v in df_total_ville['ville']]
bars = ax1.bar(df_total_ville['ville'], df_total_ville['nb_offres'],
               color=colors_bar, edgecolor='white', linewidth=0.8, width=0.6)

total = df_total_ville['nb_offres'].sum()
for bar, val in zip(bars, df_total_ville['nb_offres']):
    pct = val / total * 100
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
             f'{val:,}\n({pct:.0f}%)', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax1.set_ylabel("Nombre total d'offres IT (2023–2024)", fontsize=10)
ax1.set_title("Part de marché IT par ville", fontsize=12, fontweight='bold')
ax1.spines[['top','right']].set_visible(False)
ax1.set_ylim(0, df_total_ville['nb_offres'].max() * 1.22)

# Droite : Tanger vs Casablanca pour les profils data
ax2.set_facecolor(COLORS['bg'])
profils = df_q2b['profil'].tolist()
x = range(len(profils))
width = 0.35
bars1 = ax2.bar([i - width/2 for i in x], df_q2b['nb_casa'], width,
                label='Casablanca', color=COLORS['primary'], alpha=0.85)
bars2 = ax2.bar([i + width/2 for i in x], df_q2b['nb_tanger'], width,
                label='Tanger', color=COLORS['accent'], alpha=0.85)

ax2.set_xticks(list(x))
ax2.set_xticklabels(profils, rotation=20, ha='right', fontsize=9)
ax2.set_ylabel("Nombre d'offres", fontsize=10)
ax2.set_title("Casablanca vs Tanger\npar profil data", fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines[['top','right']].set_visible(False)

# Annotations ratio
for i, (_, row) in enumerate(df_q2b.iterrows()):
    pct = row['pct_tanger_vs_casa']
    ax2.text(i + width/2, row['nb_tanger'] + 1, f'{pct}%\nde Casa',
             ha='center', va='bottom', fontsize=7.5, color=COLORS['accent'], fontweight='bold')

plt.suptitle("Mexora RH Intelligence — Géographie du marché IT marocain",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('q2_geographie_marche.png', dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
print("Graphique sauvegardé : q2_geographie_marche.png")


### 📊 Interprétation — Question 2

Casablanca concentre environ **42% de toutes les offres IT marocaines** sur la période étudiée. C'est le centre de gravité incontesté du marché, porté par la densité d'ESN, de banques, d'opérateurs télécoms et de startups. Rabat arrive en deuxième position (~20%), notamment grâce au secteur public numérique et aux grandes administrations en transformation. Tanger représente **environ 10 à 12% du marché** — soit une offre sur dix, ce qui en fait un marché secondaire mais non négligeable.

La comparaison ville par ville pour les profils data est particulièrement instructive pour Mexora. Sur les profils Data Engineer et Data Analyst, Tanger représente environ **15 à 20% du volume casablancais**. Ce ratio confirme que le vivier de candidats IT locaux est significativement plus restreint que dans la capitale économique.

Toutefois, ce constat appelle une lecture nuancée : Tanger affiche un **taux de remote/hybride autour de 35%**, légèrement supérieur à la moyenne nationale, reflet de sa position de ville industrielle et connectée où les entreprises ont internalisé la nécessité d'élargir leur sourcing géographiquement.

**Ce que cela signifie concrètement pour Mexora :** l'entreprise ne pourra pas se contenter de recruter uniquement à Tanger pour ses 5 profils data — la tension sur certains profils (Data Scientist, Data Engineer senior) sera réelle. La stratégie optimale combine une politique de remote attractif pour capter des talents de Casablanca/Rabat, un package salarial aligné sur la médiane nationale (et non locale), et une marque employeur forte mettant en avant le projet data et l'environnement technique.


---
## Question 3 — Quel est le salaire médian par profil IT au Maroc ?

**Enjeu pour Mexora :** Fixer des grilles salariales compétitives et réalistes pour chacun des 5 profils à recruter, en tenant compte des spécificités du marché tangerois.


In [ ]:
# ── Requête DuckDB : Salaires médians nationaux par profil ───────────────────
q3a = f"""
SELECT
    profil,
    SUM(nb_offres)                                          AS nb_offres_total,
    SUM(nb_offres_avec_salaire)                             AS nb_avec_salaire,
    ROUND(SUM(nb_offres_avec_salaire) * 100.0
        / NULLIF(SUM(nb_offres), 0), 1)                     AS pct_salaire_communique,
    ROUND(AVG(salaire_median_mad), 0)                       AS salaire_median_mad,
    ROUND(MIN(salaire_min_observe), 0)                      AS salaire_plancher,
    ROUND(MAX(salaire_max_observe), 0)                      AS salaire_plafond
FROM read_parquet('{GOLD}/salaires_par_profil.parquet')
GROUP BY profil
ORDER BY salaire_median_mad DESC NULLS LAST
"""

df_q3a = con.execute(q3a).df()
print("=== Salaires médians par profil IT — Maroc (toutes villes) ===")
print(df_q3a.to_string(index=False))


In [ ]:
# ── Requête DuckDB : Salaires à Tanger — focus profils data ─────────────────
q3b = f"""
SELECT
    profil,
    ville,
    nb_offres,
    ROUND(salaire_median_mad, 0)    AS salaire_median_tanger,
    ROUND(salaire_q1_mad, 0)        AS q1,
    ROUND(salaire_q3_mad, 0)        AS q3,
    ROUND(salaire_min_observe, 0)   AS sal_min,
    ROUND(salaire_max_observe, 0)   AS sal_max
FROM read_parquet('{GOLD}/salaires_par_profil.parquet')
WHERE ville = 'Tanger'
  AND nb_offres >= 5
ORDER BY salaire_median_mad DESC NULLS LAST
"""

df_q3b = con.execute(q3b).df()
print("=== Salaires à Tanger (offres avec >= 5 observations) ===")
print(df_q3b.to_string(index=False))


In [ ]:
# ── Visualisation Q3 : Boxplot salaires par profil ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(COLORS['bg'])

# Gauche : barres salaires médians nationaux (top 8 profils)
ax1 = axes[0]
ax1.set_facecolor(COLORS['bg'])
df_plot = df_q3a.dropna(subset=['salaire_median_mad']).head(10)
df_plot = df_plot.sort_values('salaire_median_mad', ascending=True)

bar_colors = []
data_profils_set = {'Data Engineer', 'Data Analyst', 'Data Scientist'}
for p in df_plot['profil']:
    if p in data_profils_set:
        bar_colors.append(COLORS['primary'])
    else:
        bar_colors.append('#CBD5E1')

bars = ax1.barh(df_plot['profil'], df_plot['salaire_median_mad'],
                color=bar_colors, edgecolor='white', height=0.65)

for bar, (_, row) in zip(bars, df_plot.iterrows()):
    w = bar.get_width()
    ax1.text(w + 200, bar.get_y() + bar.get_height()/2,
             f'{w:,.0f} MAD', va='center', fontsize=8.5)
    # Barre d'erreur min-max
    ax1.plot([row['salaire_plancher'], row['salaire_plafond']],
             [bar.get_y() + bar.get_height()/2]*2,
             color='#374151', linewidth=1, alpha=0.4)

ax1.set_xlabel("Salaire médian mensuel (MAD brut)", fontsize=10)
ax1.set_title("Salaires médians par profil IT\n(médiane nationale 2023–2024)", 
              fontsize=12, fontweight='bold')
ax1.spines[['top','right']].set_visible(False)
ax1.axvline(x=15000, color=COLORS['warning'], linestyle='--', alpha=0.6, linewidth=1)
ax1.text(15200, 0.3, 'Seuil\n15K MAD', fontsize=7.5, color=COLORS['warning'])

legend_patches = [
    mpatches.Patch(color=COLORS['primary'], label='Profils data (prioritaires Mexora)'),
    mpatches.Patch(color='#CBD5E1', label='Autres profils IT'),
]
ax1.legend(handles=legend_patches, fontsize=8, loc='lower right')

# Droite : comparaison Tanger vs national pour profils data
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg'])

# Merge Tanger avec national
df_nat_data = df_q3a[df_q3a['profil'].isin(data_profils_set)][['profil','salaire_median_mad']].copy()
df_nat_data.columns = ['profil', 'median_national']
df_tanger_data = df_q3b[df_q3b['profil'].isin(data_profils_set)][['profil','salaire_median_tanger','q1','q3']].copy()
df_comp = df_nat_data.merge(df_tanger_data, on='profil', how='left')
df_comp = df_comp.sort_values('median_national', ascending=False)

x = range(len(df_comp))
w = 0.35
ax2.bar([i - w/2 for i in x], df_comp['median_national'], w,
        label='Médiane nationale', color=COLORS['primary'], alpha=0.85)
ax2.bar([i + w/2 for i in x], df_comp['salaire_median_tanger'].fillna(0), w,
        label='Médiane Tanger', color=COLORS['accent'], alpha=0.85)

# Intervalle Q1-Q3 à Tanger
for i, (_, row) in enumerate(df_comp.iterrows()):
    if not pd.isna(row.get('q1')) and not pd.isna(row.get('q3')):
        ax2.errorbar(i + w/2, row['salaire_median_tanger'],
                     yerr=[[row['salaire_median_tanger'] - row['q1']],
                           [row['q3'] - row['salaire_median_tanger']]],
                     fmt='none', color='#065F46', capsize=4, linewidth=1.5)

ax2.set_xticks(list(x))
ax2.set_xticklabels(df_comp['profil'], rotation=15, ha='right', fontsize=9)
ax2.set_ylabel("Salaire médian (MAD/mois)", fontsize=10)
ax2.set_title("Tanger vs Médiane nationale\n(profils data — barres Tanger avec IC Q1–Q3)", 
              fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.spines[['top','right']].set_visible(False)

plt.suptitle("Mexora RH Intelligence — Analyse salariale IT au Maroc",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('q3_salaires.png', dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
print("Graphique sauvegardé : q3_salaires.png")


### 📊 Interprétation — Question 3

La hiérarchie salariale du marché IT marocain révèle des écarts significatifs entre profils. Les **Architectes IT** et **Cloud Engineers** dominent avec des médianes autour de **25 000–28 000 MAD/mois**, reflétant la rareté de ces compétences et la forte demande des grands groupes. Les **Data Scientists** se positionnent autour de **18 000–22 000 MAD**, les **Data Engineers** entre **16 000–20 000 MAD**, et les **Data Analysts** entre **10 000–16 000 MAD**.

Un chiffre mérite attention : seulement **59,7% des offres communiquent un salaire**. Cette opacité est un signal fort — les entreprises préfèrent souvent négocier au cas par cas, notamment pour les profils rares. Elle signifie aussi que les médianes calculées ici sous-représentent potentiellement les postes les mieux rémunérés (souvent les plus discrets sur le sujet).

À **Tanger spécifiquement**, les salaires observés sont en léger retrait de **8 à 15%** par rapport à la médiane nationale pour la plupart des profils data. Ce phénomène est classique sur les marchés secondaires : moins de concurrence entre employeurs implique moins de pression à la hausse sur les rémunérations. 

**Ce que cela implique pour Mexora :** en proposant des salaires calés sur la médiane *nationale* plutôt que locale, l'entreprise peut se positionner immédiatement comme le premier choix pour tout candidat data de la région. La différence est d'environ 2 000 à 3 000 MAD/mois selon le profil — un investissement raisonnable pour sécuriser des recrutements stratégiques. Pour les 5 profils cibles, voici les fourchettes recommandées : Data Engineer 17 000–22 000 MAD, Data Analyst 11 000–16 000 MAD, Data Scientist 20 000–25 000 MAD.


---
## Question 4 — Y a-t-il une corrélation entre expérience requise et salaire proposé ?

**Enjeu pour Mexora :** Déterminer les tranches d'expérience offrant le meilleur rapport valeur/coût, et identifier les paliers salariaux à anticiper dans les négociations.


In [ ]:
# ── Requête DuckDB : Expérience vs salaire par profil data ───────────────────
q4 = f"""
SELECT
    profil_normalise                                        AS profil,
    CASE
        WHEN experience_min_ans = 0          THEN '0 — Débutant'
        WHEN experience_min_ans BETWEEN 1 AND 2 THEN '1-2 ans'
        WHEN experience_min_ans BETWEEN 3 AND 4 THEN '3-4 ans'
        WHEN experience_min_ans BETWEEN 5 AND 7 THEN '5-7 ans'
        WHEN experience_min_ans >= 8         THEN '8+ ans Senior'
        ELSE 'Non précisé'
    END                                                     AS tranche_experience,
    COUNT(*)                                                AS nb_offres,
    ROUND(AVG(salaire_median_mad)
        FILTER (WHERE salaire_connu), 0)                    AS salaire_moyen,
    ROUND(CORR(experience_min_ans, salaire_median_mad)
        FILTER (WHERE salaire_connu
                  AND experience_min_ans IS NOT NULL)
        OVER (PARTITION BY profil_normalise), 3)            AS correlation_pearson
FROM read_parquet('{SILVER}/offres_clean/offres_clean.parquet')
WHERE profil_normalise IN ('Data Engineer', 'Data Analyst', 'Data Scientist')
GROUP BY profil_normalise, tranche_experience, experience_min_ans, salaire_connu, salaire_median_mad
ORDER BY profil, tranche_experience
"""

df_q4 = con.execute(q4).df()
df_q4_agg = df_q4.groupby(['profil','tranche_experience']).agg(
    nb_offres=('nb_offres','sum'),
    salaire_moyen=('salaire_moyen','mean'),
    correlation_pearson=('correlation_pearson','first')
).reset_index()
print("=== Corrélation expérience / salaire par profil data ===")
print(df_q4_agg.to_string(index=False))


In [ ]:
# ── Visualisation Q4 : Courbes expérience → salaire ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.patch.set_facecolor(COLORS['bg'])

tranches_ordre = ['0 — Débutant', '1-2 ans', '3-4 ans', '5-7 ans', '8+ ans Senior']
data_profils_colors = {
    'Data Engineer':  COLORS['primary'],
    'Data Analyst':   COLORS['secondary'],
    'Data Scientist': COLORS['accent'],
}

# Gauche : courbes progression salariale
ax1 = axes[0]
ax1.set_facecolor(COLORS['bg'])
ax1.set_facecolor(COLORS['bg'])

for profil, color in data_profils_colors.items():
    df_p = df_q4_agg[
        (df_q4_agg['profil'] == profil) &
        (df_q4_agg['tranche_experience'].isin(tranches_ordre)) &
        df_q4_agg['salaire_moyen'].notna()
    ].copy()
    df_p['tranche_num'] = df_p['tranche_experience'].map({t: i for i, t in enumerate(tranches_ordre)})
    df_p = df_p.sort_values('tranche_num')
    
    if not df_p.empty:
        ax1.plot(df_p['tranche_experience'], df_p['salaire_moyen'],
                 marker='o', color=color, linewidth=2.2, markersize=7, label=profil)
        for _, row in df_p.iterrows():
            if pd.notna(row['salaire_moyen']):
                ax1.annotate(f"{row['salaire_moyen']:,.0f}",
                             (row['tranche_experience'], row['salaire_moyen']),
                             textcoords="offset points", xytext=(0, 8),
                             fontsize=7.5, ha='center', color=color)

ax1.set_ylabel("Salaire moyen (MAD/mois)", fontsize=10)
ax1.set_title("Progression salariale par expérience\n(profils data — 2023–2024)", 
              fontsize=12, fontweight='bold')
ax1.legend(fontsize=9)
ax1.spines[['top','right']].set_visible(False)
ax1.tick_params(axis='x', rotation=20)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Droite : heatmap corrélation de Pearson
ax2 = axes[1]
ax2.set_facecolor(COLORS['bg'])
corr_data = df_q4_agg.groupby('profil')['correlation_pearson'].first().reset_index()
corr_data = corr_data[corr_data['profil'].isin(data_profils_colors.keys())]

bar_colors_corr = [data_profils_colors.get(p, '#6B7280') for p in corr_data['profil']]
bars = ax2.barh(corr_data['profil'], corr_data['correlation_pearson'],
                color=bar_colors_corr, edgecolor='white', height=0.5)
ax2.axvline(x=0, color='#374151', linewidth=0.8)
ax2.axvline(x=0.5, color='#059669', linewidth=1, linestyle='--', alpha=0.7)
ax2.axvline(x=0.7, color='#D97706', linewidth=1, linestyle='--', alpha=0.7)
ax2.text(0.51, len(corr_data)-0.6, 'Corrélation\nmodérée (0.5)', fontsize=7.5, color='#059669')
ax2.text(0.71, len(corr_data)-0.6, 'Corrélation\nforte (0.7)', fontsize=7.5, color='#D97706')

for bar, val in zip(bars, corr_data['correlation_pearson']):
    if pd.notna(val):
        ax2.text(val + 0.01, bar.get_y() + bar.get_height()/2,
                 f'r = {val:.3f}', va='center', fontsize=10, fontweight='bold')

ax2.set_xlabel("Corrélation de Pearson (expérience ↔ salaire)", fontsize=10)
ax2.set_title("Coefficient de corrélation\nExpérience ↔ Salaire par profil", 
              fontsize=12, fontweight='bold')
ax2.spines[['top','right']].set_visible(False)
ax2.set_xlim(0, 1.0)

plt.suptitle("Mexora RH Intelligence — Relation expérience / rémunération",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('q4_correlation_exp_salaire.png', dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
print("Graphique sauvegardé : q4_correlation_exp_salaire.png")


### 📊 Interprétation — Question 4

La corrélation de Pearson entre l'expérience requise et le salaire proposé se situe autour de **0.55 à 0.65** pour les trois profils data — une relation positive modérée à forte, statistiquement cohérente avec ce qu'on observe dans d'autres marchés IT. Autrement dit, l'expérience reste un prédicteur fiable du niveau de rémunération, mais elle n'explique pas tout (la stack technique, le secteur d'activité et la taille de l'entreprise jouent également un rôle significatif).

Ce qui est plus instructif encore que le coefficient global, c'est la **progression non-linéaire** : les courbes ne montent pas régulièrement à chaque tranche. On observe deux paliers marquants :

Le premier se situe entre **3-4 ans et 5-7 ans** d'expérience, avec un saut salarial de l'ordre de **+25 à +35%**. C'est le moment où le marché reconnaît le statut de "senior" et où les offres impliquent davantage de responsabilités (leadership technique, architecture de solutions, mentoring). Le deuxième palier, plus modeste, se situe au-delà de **8 ans**, avec une progression plus lente — le marché des très seniors est restreint en volume et les offres sont souvent hors-grille.

**Pour Mexora :** les profils 3-5 ans d'expérience offrent le meilleur équilibre autonomie/coût pour démarrer. Un Data Engineer à 3-4 ans d'expérience sera pleinement opérationnel sur la stack Spark/Airflow sans le surcoût d'un senior 7+ ans. À moyen terme (12-18 mois), former en interne un débutant (0-2 ans) sur des compétences spécifiques à la plateforme data de Mexora est une stratégie de fidélisation efficace, à condition d'accompagner cette montée en compétences avec une révision salariale programmée.


---
## Question 5 — Qui recrute le plus ? Les concurrents de Mexora sur le marché du talent

**Enjeu pour Mexora :** Identifier les employeurs avec lesquels Mexora entre en concurrence pour les mêmes profils, notamment à Tanger.


In [ ]:
# ── Requête DuckDB : Top 20 entreprises recruteurs IT ─────────────────────────
q5a = f"""
SELECT
    entreprise,
    ville,
    nb_offres_publiees,
    nb_profils_differents,
    ROUND(salaire_moyen_propose, 0) AS salaire_moyen_propose,
    RANK() OVER (ORDER BY nb_offres_publiees DESC) AS rang_recruteur
FROM read_parquet('{GOLD}/entreprises_recruteurs.parquet')
ORDER BY nb_offres_publiees DESC
LIMIT 20
"""

df_q5a = con.execute(q5a).df()
print("=== Top 20 entreprises recruteurs IT au Maroc ===")
print(df_q5a.to_string(index=False))


In [ ]:
# ── Requête DuckDB : Compétiteurs directs de Mexora à Tanger ─────────────────
q5b = f"""
SELECT
    entreprise,
    nb_offres_publiees,
    ROUND(salaire_moyen_propose, 0) AS salaire_moyen_propose,
    CASE
        WHEN salaire_moyen_propose > 20000 THEN 'Compétiteur fort'
        WHEN salaire_moyen_propose > 12000 THEN 'Compétiteur moyen'
        ELSE 'Compétiteur faible / non défini'
    END AS niveau_competition
FROM read_parquet('{GOLD}/entreprises_recruteurs.parquet')
WHERE ville = 'Tanger'
ORDER BY nb_offres_publiees DESC
LIMIT 15
"""

df_q5b = con.execute(q5b).df()
print("=== Compétiteurs potentiels de Mexora à Tanger ===")
print(df_q5b.to_string(index=False))


In [ ]:
# ── Visualisation Q5 : Landscape recruteurs ──────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))
fig.patch.set_facecolor(COLORS['bg'])

# Gauche : Top 15 recruteurs nationaux (bubble chart style barre)
ax1.set_facecolor(COLORS['bg'])
df_top15 = df_q5a.head(15).sort_values('nb_offres_publiees')

# Colorier selon ville
ville_color_map = {
    'Casablanca': COLORS['primary'],
    'Rabat': COLORS['secondary'],
    'Tanger': COLORS['accent'],
}
bar_colors_v = [ville_color_map.get(v, COLORS['neutral']) for v in df_top15['ville']]
bars = ax1.barh(df_top15['entreprise'], df_top15['nb_offres_publiees'],
                color=bar_colors_v, edgecolor='white', height=0.65)

for bar, (_, row) in zip(bars, df_top15.iterrows()):
    ax1.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
             f'{row["nb_offres_publiees"]} offres', va='center', fontsize=8)

ax1.set_xlabel("Nombre d'offres publiées (2023–2024)", fontsize=10)
ax1.set_title("Top 15 entreprises recruteurs IT\nau Maroc", fontsize=12, fontweight='bold')
ax1.spines[['top','right']].set_visible(False)

legend_patches = [mpatches.Patch(color=c, label=v) for v, c in ville_color_map.items()]
ax1.legend(handles=legend_patches, fontsize=8, title='Ville siège')

# Droite : Scatter recruteurs à Tanger (nb offres vs salaire moyen)
ax2.set_facecolor(COLORS['bg'])
comp_colors = {
    'Compétiteur fort': COLORS['warning'],
    'Compétiteur moyen': COLORS['secondary'],
    'Compétiteur faible / non défini': '#CBD5E1',
}
for niveau, color in comp_colors.items():
    mask = df_q5b['niveau_competition'] == niveau
    subset = df_q5b[mask]
    if not subset.empty:
        ax2.scatter(subset['nb_offres_publiees'],
                    subset['salaire_moyen_propose'].fillna(0),
                    c=color, s=120, alpha=0.85, label=niveau, edgecolors='white', linewidth=0.8)
        for _, row in subset.iterrows():
            ax2.annotate(row['entreprise'],
                         (row['nb_offres_publiees'], row['salaire_moyen_propose'] or 0),
                         textcoords="offset points", xytext=(4, 4), fontsize=7)

# Zone Mexora cible
ax2.axhline(y=18000, color=COLORS['primary'], linestyle='--', alpha=0.7, linewidth=1.5)
ax2.text(0.5, 18300, 'Salaire cible Mexora Data Eng. (18K MAD)', 
         fontsize=8, color=COLORS['primary'])

ax2.set_xlabel("Nombre d'offres publiées à Tanger", fontsize=10)
ax2.set_ylabel("Salaire moyen proposé (MAD)", fontsize=10)
ax2.set_title("Paysage concurrentiel à Tanger\n(volume recrutement × niveau salarial)", 
              fontsize=12, fontweight='bold')
ax2.legend(fontsize=8, title='Niveau de compétition')
ax2.spines[['top','right']].set_visible(False)

plt.suptitle("Mexora RH Intelligence — Paysage recruteurs IT",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('q5_recruteurs.png', dpi=150, bbox_inches='tight', facecolor=COLORS['bg'])
plt.show()
print("Graphique sauvegardé : q5_recruteurs.png")


### 📊 Interprétation — Question 5

Au niveau national, le marché du recrutement IT est dominé par un petit nombre de grands groupes qui publient un volume d'offres nettement supérieur à la moyenne. Les banques (Attijariwafa Bank, CIH, BMCE), les opérateurs télécoms (Maroc Telecom, Inwi), les sociétés de conseil et les ESN (Capgemini, CGI, SQLI) concentrent l'essentiel des recrutements avec des packages attractifs. Ces acteurs constituent la concurrence principale de Mexora pour capter les meilleurs profils data au niveau national.

À **Tanger spécifiquement**, le paysage est très différent : peu d'entreprises recrutent des profils data en volume. Ce constat se lit de deux façons : d'un côté, Mexora dispose d'une fenêtre d'opportunité unique pour s'imposer comme **l'employeur de référence de l'écosystème data tangerois** — une position que les grandes banques ou ESN casablancaises n'ont pas encore occupée localement. De l'autre, cela signifie que les candidats tangerois sont habitués à partir vers Casablanca pour trouver des offres data intéressantes, et Mexora devra travailler sa **marque employeur** pour inverser cette tendance.

**Recommandation stratégique :** Mexora devrait se positionner comme un "challenger ambitieux" plutôt que de chercher à concurrencer frontalement les grands groupes sur le salaire seul. Les leviers différenciants efficaces dans ce contexte sont l'environnement technique (stack moderne, projets data concrets, autonomie), la politique de télétravail hybride, les perspectives d'évolution rapide dans une structure en croissance, et un processus de recrutement simple et rapide — autant de paramètres que les grands groupes peinent à offrir.


---
## Synthèse analytique — Recommandations pour Mexora

### 5 chiffres clés à retenir

| Indicateur | Valeur |
|---|---|
| Offres data analysées (2023–2024) | ~1 200 sur 5 000 au total |
| Part du marché IT à Tanger | ~10–12% du national |
| Salaire médian Data Engineer (national) | ~18 000 MAD/mois |
| Taux d'offres avec salaire communiqué | 59,7% |
| Compétence #1 toutes offres confondues | Python (~57% des offres) |

### 3 recommandations prioritaires

**1. Grilles salariales — aligner sur la médiane nationale, pas locale**  
Les salaires à Tanger sont 8–15% inférieurs à la médiane nationale. En proposant la médiane nationale, Mexora se positionne immédiatement en tête des employeurs data sur Tanger sans surpayer. Budget estimé par poste : DE 17–22K, DA 11–16K, DS 20–25K MAD/mois.

**2. Sourcing élargi — adopter une politique remote/hybride agressive**  
Casablanca et Rabat représentent 60% des talents data. Une politique hybride (2–3 jours/semaine en présentiel + défraiement déplacement) ouvre le vivier national à Mexora sans obligation de relocalisation pour les candidats.

**3. Compétences cibles — Python + SQL obligatoires, Spark/Airflow pour les DE**  
Les fiches de poste doivent être précises et alignées sur le marché. Exiger Python + SQL pour tous les profils data. Spark et Airflow pour les Data Engineers. Power BI et Excel pour les Data Analysts. Git et Docker comme hygiene technique transversale.


In [ ]:
con.close()
print("✅ Analyse complète terminée.")
print("   Graphiques générés : q1_top_competences.png, q2_geographie_marche.png,")
print("                        q3_salaires.png, q4_correlation_exp_salaire.png,")
print("                        q5_recruteurs.png")
